In [1]:
# -*- coding: utf-8 -*-
"""
Fully Learned DAE (FL-DAE) with Three-Stage RAR.

Definition:
- Stage 1: Learn phi^pm with phi-RAR. (Highest accuracy required)
- Stage 2: Learn interface h(y,t) with h-RAR, driven by learned phi.
- Stage 3: Learn inner-layer Q^pm with Q-RAR, driven by derived 2D PDE and learned phi & h.

Note: The RAR loop logic guarantees that the LAST added batch is always trained 
before breaking out of the loop.
"""

from __future__ import annotations

import math
import os
import random
import time
from typing import Dict, List, Sequence, Tuple, Union

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.optim as optim
from scipy.spatial import cKDTree
from scipy.stats import qmc

Tensor = torch.Tensor
TensorInputs = Union[Tensor, Sequence[Tensor]]

# =============================================================================
# Basic settings
# =============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
torch.backends.cudnn.benchmark = False

SEEDS = [33,99,202, 1234, 5678, 9999]
MU_LIST = [0.01, 0.001, 0.0001]

X_MIN, X_MAX = -2.0, 2.0
Y_MIN, Y_MAX = -4.0, 4.0
T_FINAL = 1.0
L_VALUE, R_VALUE = -4.0, 2.0
H0_VALUE = 0.0
XI_MAX = 12.0

# Network settings
DEPTH, WIDTH = 5, 20
LR = 1.0e-3
W_PHI_PER = 1.0
W_H_PER = 1.0
CHECK_EVERY = 100
INNER_LOSS_THRESHOLD = 1.0e-13

# -----------------------------------------------------------------------------
# Stage 1: phi-RAR Settings (Upgraded for extreme accuracy)
# -----------------------------------------------------------------------------
PHI_INITIAL_SIZE = 2600
PHI_ADD_K = 10
PHI_MAX_RAR_BATCHES = 40       # More batches for phi
PHI_CANDIDATE_SIZE = 50000     # Denser candidate search for phi
PHI_RESIDUAL_THRESHOLD = 1.0e-6 # Stricter tolerance
PHI_INNER_MAX_STEPS = 5000    # More inner training steps
N_PHI_PERIODIC = 2000

# -----------------------------------------------------------------------------
# Stage 2: h-RAR Settings
# -----------------------------------------------------------------------------
H_INITIAL_SIZE = 2600
H_ADD_K = 20
H_MAX_RAR_BATCHES = 20
H_CANDIDATE_SIZE = 35000
H_RESIDUAL_THRESHOLD = 1.0e-6
H_INNER_MAX_STEPS = 5000
N_H_PERIODIC = 1500

# -----------------------------------------------------------------------------
# Stage 3: Q-RAR Settings
# -----------------------------------------------------------------------------
Q_INITIAL_SIZE = 2600
Q_ADD_K = 20
Q_MAX_RAR_BATCHES = 20
Q_CANDIDATE_SIZE = 35000
Q_RESIDUAL_THRESHOLD = 1.0e-5
Q_INNER_MAX_STEPS = 5000
N_Q_MATCH = 2600

# Testing Settings
NUM_SAMPLES = 10000
LHS_SEED = 1234
EVAL_WARMUP = 20
EVAL_REPEAT = 200
BASE_PATH = "."
SAVE_LHS_PREDICTION = True

PI = torch.tensor(math.pi, dtype=DTYPE, device=DEVICE)

# =============================================================================
# Utilities
# =============================================================================
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sync_cuda() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def mse(x: Tensor) -> Tensor:
    return torch.mean(x.square())

def mean_std(values: Sequence[float]) -> Tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    if len(arr) <= 1:
        return float(np.nanmean(arr)), 0.0
    return float(np.nanmean(arr)), float(np.nanstd(arr, ddof=1))

def all_grads(outputs: Tensor, inputs: TensorInputs, create_graph: bool = True, retain_graph: bool | None = None):
    if retain_graph is None: retain_graph = create_graph
    single = isinstance(inputs, torch.Tensor)
    input_tuple = (inputs,) if single else tuple(inputs)
    values = torch.autograd.grad(
        outputs=outputs, inputs=input_tuple, grad_outputs=torch.ones_like(outputs),
        create_graph=create_graph, retain_graph=retain_graph, only_inputs=True, allow_unused=False
    )
    return values[0] if single else values

def sample_x(n: int) -> Tensor: return X_MIN + (X_MAX - X_MIN) * torch.rand(n, 1, device=DEVICE, dtype=DTYPE)
def sample_y(n: int) -> Tensor: return Y_MIN + (Y_MAX - Y_MIN) * torch.rand(n, 1, device=DEVICE, dtype=DTYPE)
def sample_t(n: int) -> Tensor: return T_FINAL * torch.rand(n, 1, device=DEVICE, dtype=DTYPE)
def sample_xi_m(n: int) -> Tensor: return -XI_MAX * torch.rand(n, 1, device=DEVICE, dtype=DTYPE)
def sample_xi_p(n: int) -> Tensor: return XI_MAX * torch.rand(n, 1, device=DEVICE, dtype=DTYPE)
def constant_y(value: float, n: int) -> Tensor: return torch.full((n, 1), value, device=DEVICE, dtype=DTYPE)

def source_f(x: Tensor, y: Tensor) -> Tensor:
    return torch.cos(PI * x / 4.0) * torch.cos(PI * y / 4.0)

# =============================================================================
# Neural-network model
# =============================================================================
class MLP(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, width: int, depth: int):
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(in_dim, width), nn.Tanh()]
        for _ in range(depth - 2):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, out_dim))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                init.xavier_normal_(layer.weight)
                if layer.bias is not None:
                    init.zeros_(layer.bias)

    def forward(self, z: Tensor) -> Tensor:
        return self.net(z)

class ThreeModuleModel(nn.Module):
    """Full Model: phi, h, and Q are ALL learned."""
    def __init__(self):
        super().__init__()
        self.N_phi = MLP(2, 2, WIDTH, DEPTH)
        self.N_h = MLP(2, 1, WIDTH, DEPTH)
        self.N_Q = MLP(4, 2, WIDTH, DEPTH)

    def phi_m(self, x: Tensor, y: Tensor) -> Tensor:
        raw = self.N_phi(torch.cat([x, y], dim=1))[:, 0:1]
        return L_VALUE + (x - X_MIN) * raw

    def phi_p(self, x: Tensor, y: Tensor) -> Tensor:
        raw = self.N_phi(torch.cat([x, y], dim=1))[:, 1:2]
        return R_VALUE + (x - X_MAX) * raw

    def h(self, y: Tensor, t: Tensor) -> Tensor:
        return H0_VALUE + t * self.N_h(torch.cat([y, t], dim=1))

    def Q_m(self, xi: Tensor, h: Tensor, y: Tensor, t: Tensor) -> Tensor:
        raw = self.N_Q(torch.cat([xi, h, y, t], dim=1))[:, 0:1]
        return ((xi + XI_MAX) / XI_MAX) * raw

    def Q_p(self, xi: Tensor, h: Tensor, y: Tensor, t: Tensor) -> Tensor:
        raw = self.N_Q(torch.cat([xi, h, y, t], dim=1))[:, 1:2]
        return ((XI_MAX - xi) / XI_MAX) * raw

def set_trainable(model: ThreeModuleModel, phi: bool, h: bool, Q: bool) -> None:
    for p in model.N_phi.parameters(): p.requires_grad_(phi)
    for p in model.N_h.parameters(): p.requires_grad_(h)
    for p in model.N_Q.parameters(): p.requires_grad_(Q)

# =============================================================================
# Stage 1: phi-RAR
# =============================================================================
def outer_loss_split(model: ThreeModuleModel, x_m: Tensor, y_m: Tensor, x_p: Tensor, y_p: Tensor, x_per: Tensor):
    pm = model.phi_m(x_m, y_m)
    pm_x, pm_y = all_grads(pm, (x_m, y_m), create_graph=True, retain_graph=True)
    rm = pm * (pm_x + pm_y) - source_f(x_m, y_m)

    pp = model.phi_p(x_p, y_p)
    pp_x, pp_y = all_grads(pp, (x_p, y_p), create_graph=True, retain_graph=True)
    rp = pp * (pp_x + pp_y) - source_f(x_p, y_p)

    loss_res = mse(rm) + mse(rp)
    y_l, y_r = constant_y(Y_MIN, x_per.shape[0]), constant_y(Y_MAX, x_per.shape[0])
    loss_per = mse(model.phi_m(x_per, y_l) - model.phi_m(x_per, y_r)) + mse(model.phi_p(x_per, y_l) - model.phi_p(x_per, y_r))
    return loss_res + W_PHI_PER * loss_per

def phi_residual_side(model: ThreeModuleModel, side: str, x_base: Tensor, y_base: Tensor) -> Tensor:
    x, y = x_base.detach().clone().requires_grad_(True), y_base.detach().clone().requires_grad_(True)
    phi = model.phi_m(x, y) if side == "m" else model.phi_p(x, y)
    phi_x, phi_y = all_grads(phi, (x, y), create_graph=False, retain_graph=False)
    return (phi.detach() * (phi_x.detach() + phi_y.detach()) - source_f(x, y)).detach()

def train_phi_rar(model: ThreeModuleModel, x_periodic: Tensor) -> Dict[str, object]:
    set_trainable(model, phi=True, h=False, Q=False)
    optimizer = optim.Adam(model.N_phi.parameters(), lr=LR)

    x_m, y_m = sample_x(PHI_INITIAL_SIZE), sample_y(PHI_INITIAL_SIZE)
    x_p, y_p = sample_x(PHI_INITIAL_SIZE), sample_y(PHI_INITIAL_SIZE)

    batches, total_added = 0, 0
    trace: List[float] = []
    sync_cuda()
    start = time.perf_counter()

    while True:
        # =====================================================================
        # 内部训练循环
        # 注意：此处不仅训练初始点，如果 batches > 0，也会彻底训练刚刚加入的那批残差点
        # =====================================================================
        n_inner = 0
        loss_value = float("inf")
        model.train()
        while n_inner < PHI_INNER_MAX_STEPS and loss_value >= INNER_LOSS_THRESHOLD:
            optimizer.zero_grad(set_to_none=True)
            xm, ym = x_m.detach().clone().requires_grad_(True), y_m.detach().clone().requires_grad_(True)
            xp, yp = x_p.detach().clone().requires_grad_(True), y_p.detach().clone().requires_grad_(True)
            
            loss = outer_loss_split(model, xm, ym, xp, yp, x_periodic)
            loss.backward()
            optimizer.step()

            n_inner += 1
            if n_inner % CHECK_EVERY == 0 or n_inner == PHI_INNER_MAX_STEPS:
                loss_value = loss.item()
                trace.append(loss_value)

        # 保证最后一批被训练后，再判定是否退出
        if batches >= PHI_MAX_RAR_BATCHES:
            print("   [phi-RAR] Max batches reached and final batch trained. Stopping.")
            break

        # 寻找残差点
        model.eval()
        x_cm, y_cm = sample_x(PHI_CANDIDATE_SIZE), sample_y(PHI_CANDIDATE_SIZE)
        x_cp, y_cp = sample_x(PHI_CANDIDATE_SIZE), sample_y(PHI_CANDIDATE_SIZE)

        with torch.enable_grad():
            r_m = phi_residual_side(model, "m", x_cm, y_cm).abs().reshape(-1)
            r_p = phi_residual_side(model, "p", x_cp, y_cp).abs().reshape(-1)

        last_mean = 0.5 * (float(r_m.mean().item()) + float(r_p.mean().item()))
        print(f"   [phi-RAR] batch={batches + 1:2d}, mean residual={last_mean:.3e}, points/side={x_m.shape[0]}")

        if last_mean < PHI_RESIDUAL_THRESHOLD:
            print("   [phi-RAR] mean residual tolerance reached.")
            break

        idx_m = torch.topk(r_m, k=PHI_ADD_K, largest=True).indices
        idx_p = torch.topk(r_p, k=PHI_ADD_K, largest=True).indices

        x_m = torch.cat([x_m, x_cm[idx_m].detach()], dim=0)
        y_m = torch.cat([y_m, y_cm[idx_m].detach()], dim=0)
        x_p = torch.cat([x_p, x_cp[idx_p].detach()], dim=0)
        y_p = torch.cat([y_p, y_cp[idx_p].detach()], dim=0)
        batches += 1

    sync_cuda()
    return {"time": time.perf_counter() - start, "batches": batches, "final_loss": trace[-1], "trace": np.asarray(trace, dtype=np.float64)}

# =============================================================================
# Stage 2: h-RAR
# =============================================================================
def interface_residual(model: ThreeModuleModel, y_t: Tensor, create_graph: bool) -> Tensor:
    y, t = y_t[:, 0:1], y_t[:, 1:2]
    h = model.h(y, t)
    grads = all_grads(h.sum(), y_t, create_graph=create_graph, retain_graph=create_graph)
    h_y, h_t = grads[:, 0:1], grads[:, 1:2]

    if create_graph:
        phi_m, phi_p = model.phi_m(h, y), model.phi_p(h, y)
        return h_t - 0.5 * (h_y - 1.0) * (phi_m + phi_p)

    with torch.no_grad():
        h_v, y_v = h.detach(), y.detach()
        phi_m, phi_p = model.phi_m(h_v, y_v), model.phi_p(h_v, y_v)
        return (h_t.detach() - 0.5 * (h_y.detach() - 1.0) * (phi_m + phi_p)).detach()

def train_h_rar(model: ThreeModuleModel, t_boundary: Tensor) -> Dict[str, object]:
    set_trainable(model, phi=False, h=True, Q=False)
    optimizer = optim.Adam(model.N_h.parameters(), lr=LR)

    y0, t0 = sample_y(H_INITIAL_SIZE), sample_t(H_INITIAL_SIZE)
    internal_batch = torch.cat([y0, t0], dim=1).detach()

    batches = 0
    trace: List[float] = []
    sync_cuda()
    start = time.perf_counter()

    while True:
        n_inner = 0
        loss_value = float("inf")
        model.train()
        while n_inner < H_INNER_MAX_STEPS and loss_value >= INNER_LOSS_THRESHOLD:
            optimizer.zero_grad(set_to_none=True)
            current_internal = internal_batch.detach().clone().requires_grad_(True)
            
            res = interface_residual(model, current_internal, create_graph=True)
            y_l, y_r = constant_y(Y_MIN, t_boundary.shape[0]), constant_y(Y_MAX, t_boundary.shape[0])
            loss = mse(res) + W_H_PER * mse(model.h(y_l, t_boundary) - model.h(y_r, t_boundary))
            
            loss.backward()
            optimizer.step()

            n_inner += 1
            if n_inner % CHECK_EVERY == 0 or n_inner == H_INNER_MAX_STEPS:
                loss_value = loss.item()
                trace.append(loss_value)

        # 保证最后一批被训练后，再判定是否退出
        if batches >= H_MAX_RAR_BATCHES:
            break

        model.eval()
        y_c, t_c = sample_y(H_CANDIDATE_SIZE), sample_t(H_CANDIDATE_SIZE)
        candidate = torch.cat([y_c, t_c], dim=1).requires_grad_(True)

        with torch.enable_grad():
            abs_residual = interface_residual(model, candidate, create_graph=False).abs().reshape(-1)
        
        last_mean = float(abs_residual.mean().item())
        print(f"   [h-RAR] batch={batches + 1:2d}, mean residual={last_mean:.3e}, internal points={internal_batch.shape[0]}")

        if last_mean < H_RESIDUAL_THRESHOLD:
            break

        topk = torch.topk(abs_residual, k=min(H_ADD_K, abs_residual.numel()), largest=True).indices
        internal_batch = torch.cat([internal_batch, candidate[topk].detach()], dim=0).detach()
        batches += 1

    sync_cuda()
    return {"time": time.perf_counter() - start, "batches": batches, "final_loss": trace[-1], "trace": np.asarray(trace, dtype=np.float64)}

# =============================================================================
# Stage 3: Q-RAR
# =============================================================================
def frozen_h_data(model: ThreeModuleModel, y_base: Tensor, t_base: Tensor):
    y, t = y_base.detach().clone().requires_grad_(True), t_base.detach().clone().requires_grad_(True)
    h_val = model.h(y, t)
    h_y, h_t = all_grads(h_val, (y, t), create_graph=False, retain_graph=False)
    return h_val.detach(), h_y.detach(), h_t.detach()

def q_residual_frozen(model: ThreeModuleModel, side: str, xi_base: Tensor, y_base: Tensor, t_base: Tensor, training_graph: bool) -> Tensor:
    h, h_y, h_t = frozen_h_data(model, y_base, t_base)
    y, t = y_base.detach(), t_base.detach()
    # 使用学习到的 phi
    phi_val = (model.phi_m(h, y) if side == "m" else model.phi_p(h, y)).detach()
    metric = torch.sqrt(1.0 + h_y.square())

    q_input = torch.cat([xi_base.detach(), h, y, t], dim=1).detach().requires_grad_(True)
    xi, h_q, y_q, t_q = q_input[:, 0:1], q_input[:, 1:2], q_input[:, 2:3], q_input[:, 3:4]
    Q = model.Q_m(xi, h_q, y_q, t_q) if side == "m" else model.Q_p(xi, h_q, y_q, t_q)

    q_grad = all_grads(Q, q_input, create_graph=True, retain_graph=True)
    Q_xi = q_grad[:, 0:1]
    Q_xixi = all_grads(Q_xi, q_input, create_graph=training_graph, retain_graph=training_graph)[:, 0:1]

    coeff = (h_t + (phi_val + Q) * (1.0 - h_y)) / metric
    residual = Q_xixi + coeff * Q_xi
    return residual if training_graph else residual.detach()

def q_match_frozen(model: ThreeModuleModel, y_base: Tensor, t_base: Tensor) -> Tensor:
    h, _, _ = frozen_h_data(model, y_base, t_base)
    y, t = y_base.detach(), t_base.detach()
    pm_val = model.phi_m(h, y).detach()
    pp_val = model.phi_p(h, y).detach()
    middle = 0.5 * (pm_val + pp_val)
    xi0 = torch.zeros_like(t)
    qm, qp = model.Q_m(xi0, h, y, t), model.Q_p(xi0, h, y, t)
    return mse(pm_val + qm - middle) + mse(pp_val + qp - middle)

def train_Q_rar(model: ThreeModuleModel) -> Dict[str, object]:
    set_trainable(model, phi=False, h=False, Q=True)
    optimizer = optim.Adam(model.N_Q.parameters(), lr=LR)

    xi_m, y_m, t_m = sample_xi_m(Q_INITIAL_SIZE), sample_y(Q_INITIAL_SIZE), sample_t(Q_INITIAL_SIZE)
    xi_p, y_p, t_p = sample_xi_p(Q_INITIAL_SIZE), sample_y(Q_INITIAL_SIZE), sample_t(Q_INITIAL_SIZE)
    y_match, t_match = sample_y(N_Q_MATCH), sample_t(N_Q_MATCH)

    batches = 0
    trace: List[float] = []
    sync_cuda()
    start = time.perf_counter()

    while True:
        n_inner = 0
        loss_value = float("inf")
        model.train()
        while n_inner < Q_INNER_MAX_STEPS and loss_value >= INNER_LOSS_THRESHOLD:
            optimizer.zero_grad(set_to_none=True)
            rm = q_residual_frozen(model, "m", xi_m, y_m, t_m, training_graph=True)
            rp = q_residual_frozen(model, "p", xi_p, y_p, t_p, training_graph=True)
            loss_match = q_match_frozen(model, y_match, t_match)
            
            loss = mse(rm) + mse(rp) + loss_match
            loss.backward()
            optimizer.step()

            n_inner += 1
            if n_inner % CHECK_EVERY == 0 or n_inner == Q_INNER_MAX_STEPS:
                loss_value = loss.item()
                trace.append(loss_value)

        # 保证最后一批被训练后，再判定是否退出
        if batches >= Q_MAX_RAR_BATCHES:
            break

        model.eval()
        xi_cm, y_cm, t_cm = sample_xi_m(Q_CANDIDATE_SIZE), sample_y(Q_CANDIDATE_SIZE), sample_t(Q_CANDIDATE_SIZE)
        xi_cp, y_cp, t_cp = sample_xi_p(Q_CANDIDATE_SIZE), sample_y(Q_CANDIDATE_SIZE), sample_t(Q_CANDIDATE_SIZE)

        with torch.enable_grad():
            rm_cand = q_residual_frozen(model, "m", xi_cm, y_cm, t_cm, training_graph=False).abs().reshape(-1)
            rp_cand = q_residual_frozen(model, "p", xi_cp, y_cp, t_cp, training_graph=False).abs().reshape(-1)

        last_mean = 0.5 * (float(rm_cand.mean().item()) + float(rp_cand.mean().item()))
        print(f"   [Q-RAR] batch={batches + 1:2d}, mean residual={last_mean:.3e}, points/side={xi_m.shape[0]}")

        if last_mean < Q_RESIDUAL_THRESHOLD:
            break

        topk_m = torch.topk(rm_cand, k=min(Q_ADD_K, rm_cand.numel()), largest=True).indices
        topk_p = torch.topk(rp_cand, k=min(Q_ADD_K, rp_cand.numel()), largest=True).indices

        xi_m = torch.cat([xi_m, xi_cm[topk_m].detach()], dim=0)
        y_m = torch.cat([y_m, y_cm[topk_m].detach()], dim=0)
        t_m = torch.cat([t_m, t_cm[topk_m].detach()], dim=0)
        xi_p = torch.cat([xi_p, xi_cp[topk_p].detach()], dim=0)
        y_p = torch.cat([y_p, y_cp[topk_p].detach()], dim=0)
        t_p = torch.cat([t_p, t_cp[topk_p].detach()], dim=0)
        
        batches += 1

    sync_cuda()
    return {"time": time.perf_counter() - start, "batches": batches, "final_loss": trace[-1], "trace": np.asarray(trace, dtype=np.float64)}

# =============================================================================
# Reconstruction & Testing
# =============================================================================
def reconstruct_FLDAE(model: ThreeModuleModel, x: Tensor, y: Tensor, t: Tensor, mu: float) -> Tensor:
    with torch.enable_grad():
        y_in, t_in = y.detach().clone().requires_grad_(True), t.detach()
        h_val = model.h(y_in, t_in)
        h_y = all_grads(h_val, y_in, create_graph=False, retain_graph=False)

    with torch.no_grad():
        h = h_val.detach()
        metric = torch.sqrt(1.0 + h_y.detach().square())
        xi = (x - h) * metric / float(mu)
        xi_m, xi_p = torch.clamp(xi, min=-XI_MAX, max=0.0), torch.clamp(xi, min=0.0, max=XI_MAX)
        
        pm, pp = model.phi_m(x, y), model.phi_p(x, y)
        qm, qp = model.Q_m(xi_m, h, y, t), model.Q_p(xi_p, h, y, t)
        return torch.where(x <= h, pm + qm, pp + qp)

def get_target_col(df: pd.DataFrame) -> str:
    return "u" if "u" in df.columns else ("u0" if "u0" in df.columns else str(df.columns[-1]))

def true_solution_filename(mu: float) -> str:
    return f"2d_U0_all_t_u_x_y_t_mu{int(round(-math.log10(mu)))}_101_101_101_Mathematica_620.csv"

def build_or_load_lhs_test_set(mu: float) -> Dict[str, object]:
    df_true = pd.read_csv(os.path.join(BASE_PATH, true_solution_filename(mu)))
    df_true.columns = [str(col).lower().strip() for col in df_true.columns]
    df_true = df_true.sort_values(by=["t", "x", "y"]).reset_index(drop=True)
    
    index_file = f"2d_LHS_sample_indices_mu{mu:.0e}.npy"
    if os.path.exists(index_file):
        sample_indices = np.load(index_file).astype(np.int64)
    else:
        tree = cKDTree(df_true[["t", "x", "y"]].to_numpy(dtype=np.float64))
        mins, maxs = [df_true[c].min() for c in ["t","x","y"]], [df_true[c].max() for c in ["t","x","y"]]
        selected, used = [], set()
        for batch_id in range(100):
            if len(selected) >= NUM_SAMPLES: break
            scaled = qmc.scale(qmc.LatinHypercube(d=3, seed=LHS_SEED + batch_id).random(NUM_SAMPLES), mins, maxs)
            for idx in tree.query(scaled)[1]:
                if int(idx) not in used:
                    used.add(int(idx)); selected.append(int(idx))
                    if len(selected) == NUM_SAMPLES: break
        if len(selected) < NUM_SAMPLES:
            rem = np.setdiff1d(np.arange(len(df_true)), np.asarray(selected))
            selected.extend(np.random.default_rng(LHS_SEED).choice(rem, size=NUM_SAMPLES - len(selected), replace=False))
        sample_indices = np.asarray(selected, dtype=np.int64)
        np.save(index_file, sample_indices)

    chosen = df_true.iloc[sample_indices]
    t_np, x_np, y_np = chosen["t"].to_numpy().reshape(-1, 1), chosen["x"].to_numpy().reshape(-1, 1), chosen["y"].to_numpy().reshape(-1, 1)
    return {
        "t": torch.tensor(t_np, dtype=DTYPE, device=DEVICE), "x": torch.tensor(x_np, dtype=DTYPE, device=DEVICE),
        "y": torch.tensor(y_np, dtype=DTYPE, device=DEVICE), "true": chosen[get_target_col(df_true)].to_numpy().reshape(-1),
        "t_np": t_np, "x_np": x_np, "y_np": y_np, "n_test": len(sample_indices),
    }

def timed_evaluation(model: ThreeModuleModel, data: Dict[str, object], mu: float):
    x, y, t = data["x"], data["y"], data["t"]
    for _ in range(EVAL_WARMUP): _ = reconstruct_FLDAE(model, x, y, t, mu)
    sync_cuda()
    start = time.perf_counter()
    for _ in range(EVAL_REPEAT): _ = reconstruct_FLDAE(model, x, y, t, mu)
    sync_cuda()
    t_eval = (time.perf_counter() - start) / EVAL_REPEAT
    pred_v = reconstruct_FLDAE(model, x, y, t, mu).detach().cpu().numpy().reshape(-1)
    true_v = data["true"]
    diff = pred_v - true_v
    return t_eval, pred_v, float(np.linalg.norm(diff) / np.linalg.norm(true_v)), float(np.max(np.abs(diff)))

# =============================================================================
# Main benchmark
# =============================================================================
def main() -> None:
    print("\n" + "=" * 92)
    print("Fully Learned DAE: Learned phi + Learned h + Learned Q (ALL WITH RAR)")
    print(f"Device={DEVICE} | N_test={NUM_SAMPLES} | LHS seed={LHS_SEED}")
    print("=" * 92 + "\n")

    lhs_data = {mu: build_or_load_lhs_test_set(mu) for mu in MU_LIST}
    metrics: Dict[float, List[Dict[str, object]]] = {mu: [] for mu in MU_LIST}

    for seed in SEEDS:
        print(f"\n--- Running seed={seed} ---")
        set_seed(seed)
        model = ThreeModuleModel().to(DEVICE)

        # Stage 1: phi-RAR (Most crucial foundation)
        phi_stats = train_phi_rar(model, sample_x(N_PHI_PERIODIC))

        # Stage 2: h-RAR
        h_stats = train_h_rar(model, sample_t(N_H_PERIODIC))

        # Stage 3: Q-RAR
        Q_stats = train_Q_rar(model)

        t_train = float(phi_stats["time"] + h_stats["time"] + Q_stats["time"])
        print(f" > trained: T_train={t_train:.2f}s, phi_batches={phi_stats['batches']}, h_batches={h_stats['batches']}, Q_batches={Q_stats['batches']}")

        model.eval()
        set_trainable(model, phi=False, h=False, Q=False)

        for mu in MU_LIST:
            data = lhs_data[mu]
            t_eval, pred, e2, einf = timed_evaluation(model, data, mu)
            print(f"    -> [mu={mu}] T_eval={t_eval:.6e}s, e2={e2:.3e}, einf={einf:.3e}")

            if SAVE_LHS_PREDICTION:
                pd.DataFrame({
                    "t": data["t_np"].reshape(-1), "x": data["x_np"].reshape(-1), "y": data["y_np"].reshape(-1), "u": pred,
                }).to_csv(f"2d_FLDAE_U0_predicted_LHS_mu{mu:.0e}_seed{seed}.csv", index=False)

            metrics[mu].append({
                "Seed": seed, "N_test": int(data["n_test"]),
                "e2": e2, "einf": einf, "T_train": t_train, "T_eval": t_eval
            })

    print("\n" + "=" * 92)
    for mu in MU_LIST:
        df = pd.DataFrame(metrics[mu])
        df.to_csv(f"2d_FLDAE_mu{mu:.0e}_Metrics_Summary2.csv", index=False)
        stats = {name: mean_std(df[name].values) for name in ["e2", "einf", "T_train", "T_eval"]}
        print(f"\n### FL-DAE, mu={mu} ###")
        print(f"e2: {stats['e2'][0]:.3e} +/- {stats['e2'][1]:.3e}")
        print(f"einf: {stats['einf'][0]:.3e} +/- {stats['einf'][1]:.3e}")

if __name__ == "__main__":
    main()


Fully Learned DAE: Learned phi + Learned h + Learned Q (ALL WITH RAR)
Device=cuda | N_test=10000 | LHS seed=1234


--- Running seed=33 ---
   [phi-RAR] batch= 1, mean residual=3.378e-02, points/side=2600
   [phi-RAR] batch= 2, mean residual=1.365e-02, points/side=2610
   [phi-RAR] batch= 3, mean residual=7.164e-03, points/side=2620
   [phi-RAR] batch= 4, mean residual=1.271e-02, points/side=2630
   [phi-RAR] batch= 5, mean residual=4.779e-03, points/side=2640
   [phi-RAR] batch= 6, mean residual=4.185e-03, points/side=2650
   [phi-RAR] batch= 7, mean residual=7.566e-03, points/side=2660
   [phi-RAR] batch= 8, mean residual=4.728e-03, points/side=2670
   [phi-RAR] batch= 9, mean residual=4.122e-03, points/side=2680
   [phi-RAR] batch=10, mean residual=2.952e-03, points/side=2690
   [phi-RAR] batch=11, mean residual=2.754e-03, points/side=2700
   [phi-RAR] batch=12, mean residual=2.687e-03, points/side=2710
   [phi-RAR] batch=13, mean residual=5.215e-03, points/side=2720
   [phi-RAR] ba